# 09.01 · Tool / function calling

**Chapter:** Tool Use & Agents · **Milestone:** M4

## Goals
- [ ] How tool calling works: the model emits a structured *request*; **your code** runs the function
- [ ] Define tools as JSON Schema — by hand and generated from Python functions with Pydantic
- [ ] Handle parallel calls, `tool_choice`, bad arguments, unknown tools and exceptions
- [ ] Translate between provider formats (OpenAI, Anthropic, Gemini)

## Resources
- [OpenAI – Function calling](https://platform.openai.com/docs/guides/function-calling) — free
- [Anthropic – Tool use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview) — free
- [Gemini – Function calling](https://ai.google.dev/gemini-api/docs/function-calling) — free

The model never executes anything. It returns "please call `get_weather` with `{"city": "Lima"}`"; you decide whether and how to run it,
then send the result back. That boundary is where all your safety and validation lives.

In [ ]:
import inspect
import json

from pydantic import BaseModel, Field, TypeAdapter, ValidationError

from aieng.llm import client

c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}

## 1. One tool, one round trip

In [ ]:
WEATHER = {"lima": "19°C, cloudy", "cusco": "12°C, sunny", "arequipa": "22°C, clear"}


def get_weather(city: str) -> str:
    return WEATHER.get(city.lower(), f"no data for {city}")


tools = [{"type": "function", "function": {
    "name": "get_weather",
    "description": "Current weather for a city in Peru.",
    "parameters": {"type": "object", "properties": {"city": {"type": "string", "description": "city name, e.g. Lima"}}, "required": ["city"]},
}}]

messages = [{"role": "user", "content": "Should I bring a jacket to Cusco today?"}]
r = c.chat.completions.create(model=MODEL, messages=messages, tools=tools, **EXTRA)
msg = r.choices[0].message
print("finish_reason:", r.choices[0].finish_reason)
print("tool calls:", [(t.id, t.function.name, t.function.arguments) for t in msg.tool_calls or []])

Run it, send the result back as a `tool` message (linked by `tool_call_id`), and let the model finish.

In [ ]:
messages.append(msg.model_dump(exclude_none=True))  # the assistant's tool-call message must stay in the history
for call in msg.tool_calls:
    result = get_weather(**json.loads(call.function.arguments))
    messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

final = c.chat.completions.create(model=MODEL, messages=messages, tools=tools, **EXTRA)
print(final.choices[0].message.content)

## 2. Schemas from Python functions
Hand-written JSON Schema drifts from the code. Generate it from type hints + docstrings instead (this is what every framework does).

In [ ]:
def tool_schema(fn):
    """OpenAI tool definition from a typed Python function."""
    params = {name: (p.annotation, ... if p.default is inspect.Parameter.empty else p.default)
              for name, p in inspect.signature(fn).parameters.items()}
    model = type(f"{fn.__name__}_args", (BaseModel,), {"__annotations__": {k: v[0] for k, v in params.items()},
                                                       **{k: v[1] for k, v in params.items() if v[1] is not ...}})
    schema = model.model_json_schema()
    schema.pop("title", None)
    return {"type": "function", "function": {"name": fn.__name__, "description": inspect.getdoc(fn), "parameters": schema}}


def convert_currency(amount: float, from_currency: str, to_currency: str = "USD") -> str:
    """Convert money between currencies (PEN, USD, EUR) using today's rate."""
    rates = {"PEN": 0.27, "USD": 1.0, "EUR": 1.08}  # to USD; illustrative
    return f"{amount * rates[from_currency.upper()] / rates[to_currency.upper()]:.2f} {to_currency.upper()}"


print(json.dumps(tool_schema(convert_currency), indent=1))

## 3. A robust dispatcher
Things that go wrong: unknown tool names, malformed JSON, wrong types, exceptions inside the tool. **Don't crash: return the error to the
model as the tool result** — it usually fixes its own call on the next step. Validate arguments with Pydantic before running anything.

In [ ]:
REGISTRY = {fn.__name__: fn for fn in [get_weather, convert_currency]}
TOOLS = [tool_schema(fn) for fn in REGISTRY.values()]


def dispatch(call):
    fn = REGISTRY.get(call.function.name)
    if fn is None:
        return f"Error: unknown tool {call.function.name!r}. Available: {list(REGISTRY)}"
    try:
        raw = json.loads(call.function.arguments or "{}")
        hints = {k: v.annotation for k, v in inspect.signature(fn).parameters.items()}
        args = {k: TypeAdapter(hints[k]).validate_python(v) for k, v in raw.items() if k in hints}
        return str(fn(**args))
    except (json.JSONDecodeError, ValidationError, KeyError, TypeError) as e:
        return f"Error calling {fn.__name__}: {type(e).__name__}: {e}"


class FakeCall:  # mimic the SDK objects to test the dispatcher without an LLM
    def __init__(self, name, arguments):
        self.function = type("F", (), {"name": name, "arguments": arguments})()


print(dispatch(FakeCall("convert_currency", '{"amount": "100", "from_currency": "pen"}')))  # "100" coerced to float
print(dispatch(FakeCall("convert_currency", '{"amount": 100, "from_currency": "JPY"}')))  # KeyError -> message
print(dispatch(FakeCall("launch_rocket", "{}")))
print(dispatch(FakeCall("get_weather", "{city: Lima}")))  # malformed JSON

## 4. Parallel calls and `tool_choice`
Models can request several calls in one turn. `tool_choice` controls whether they may/must call tools.

In [ ]:
def run_turn(question, tool_choice="auto"):
    messages = [{"role": "user", "content": question}]
    r = c.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, tool_choice=tool_choice, **EXTRA)
    msg = r.choices[0].message
    if not msg.tool_calls:
        return msg.content, []
    messages.append(msg.model_dump(exclude_none=True))
    results = []
    for call in msg.tool_calls:
        out = dispatch(call)
        results.append((call.function.name, call.function.arguments, out))
        messages.append({"role": "tool", "tool_call_id": call.id, "content": out})
    return c.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, **EXTRA).choices[0].message.content, results


for q, choice in [("Weather in Lima and Arequipa, and how much is 250 PEN in EUR?", "auto"),
                  ("Tell me a joke.", "auto"),
                  ("Tell me a joke.", {"type": "function", "function": {"name": "get_weather"}})]:
    answer, calls = run_turn(q, choice)
    print(f"Q: {q} | tool_choice={choice if isinstance(choice, str) else 'get_weather'}\n  calls: {calls}\n  A: {answer[:150]}\n")

Local Ollama **ignores `tool_choice`** (hosted APIs honour it), so the forced call above may not happen locally. Portable fallback used in
later lessons: if a required tool wasn't called, append "Use the tools before answering." and retry.

## 5. Provider formats
Same concept, different JSON. Frameworks and gateways (LiteLLM) translate for you; know the shapes to debug.
```python
# OpenAI / OpenAI-compatible (Ollama, Groq, Gemini-compat, vLLM...)
{"type": "function", "function": {"name": ..., "description": ..., "parameters": SCHEMA}}
# → response: message.tool_calls[i].function.arguments (a JSON *string*); reply with role="tool", tool_call_id

# Anthropic Messages API
{"name": ..., "description": ..., "input_schema": SCHEMA}
# → response: content block {"type": "tool_use", "id", "name", "input": {...dict}}; reply with a user message containing
#   {"type": "tool_result", "tool_use_id": ..., "content": ...}

# Gemini native (google-genai)
types.Tool(function_declarations=[{"name": ..., "description": ..., "parameters": SCHEMA}])
# → response: part.function_call (name, args dict); reply with types.Part.from_function_response(name=..., response={...})
```

In [ ]:
def to_anthropic(openai_tool):
    f = openai_tool["function"]
    return {"name": f["name"], "description": f["description"], "input_schema": f["parameters"]}


print(json.dumps(to_anthropic(TOOLS[1]), indent=1)[:300])

## 6. Designing good tools
- **Names and descriptions are prompts.** Say what it does, when to use it, what it returns, units and formats.
- **Few, well-scoped tools** beat many overlapping ones (each definition also costs tokens on every call, 06.04).
- **Return compact, model-friendly output** (not 5,000-line JSON); include ids the model can pass to other tools.
- **Make errors actionable**: "city not found; valid cities: Lima, Cusco" lets the model recover.
- **Side effects need guards**: confirmation, idempotency keys, dry-run modes (09.06).

## Exercises
1. Add a `calculator(expression: str)` tool that evaluates arithmetic **safely** (parse with `ast`, allow only numbers and + - * / ** ( )). Never `eval` model output.
2. Send the same 3-tool request through the native Anthropic or Gemini SDK (if you have a key) using `to_anthropic` / a Gemini converter.
3. Write 10 test prompts and measure: right tool chosen? valid arguments? Compare `qwen3:1.7b` with a hosted model. (This is a tool-use eval — chapter 10.)

In [ ]:
# your code here